# VQVAE 架构解析

VQVAE 的核心架构如下图所示:

<div align="center">
    <img src="./imgs/VQVAE_architecture.png" alt="VQVAE Architecture" style="width: 650px; height: auto;">
</div>

本篇notebook基于如下链接的内容和代码进行整理，参考链接：

[参考知乎解析](https://zhuanlan.zhihu.com/p/633744455)

[VQVAE 论文](https://arxiv.org/pdf/1711.00937)

[VQVAE 模型 Pytorch 实现](https://github.com/SingleZombie/DL-Demos/tree/master/dldemos/VQVAE)

## VQVAE 的引入

VAE 的生成图像的质量并不是很好，VQVAE 的作者认为，是因为 VAE 把图像编码成了连续的向量，但是我们知道，图像的像素值**并不是连续的，而是离散的**，因此编码成**离散向量**会更符合图像的特征，就类似于你说某个班级的人数，那肯定是整数，而不是说一个小数(比如11.4个人)，要么11个要么12个，你的人怎么还少了0.6个？(~~截肢了~~)图像也差不多是一样的道理

为了把图像编码为离散的向量，我们可以借鉴一下 NLP 模型的处理方式: token embedding，其实就是一个“词典”，每个词对应一个特征向量，把句子中的每个词都使用这个“词典”进行替换，就得到了[seq_len, dim]的特征了；

我们也可以在 VQVAE 中应用类似的查表 embedding 的方式映射到嵌入空间中，这个类似“词典”的编码表就称作 **codebook**

我们进行离散编码了之后，还有一个问题就是: 我们怎么采样？那就要使用我们上篇提到的 **PixelCNN** 模型了，它能够你和一个离散的分布，在原始的实现中，他就给每个像素输出0~255的一共256个logit构成概率分布，就可以进行采样了，接下来我们将一步步拆解 VQVAE

## 离散编码的生成与梯度传递

### 离散编码的生成

对于一个形状为[num_embedding, hidden_dim]的 codebook 和 形状为 [hidden_dim, h, w]的图像 $z_e$ (已映射到隐空间)，我们要把 h*w 个向量替换为 codebook 的向量，就需要这样做:

对于图像的每个向量$z_e$，计算它和 codebook 中所有向量的欧氏距离，并取距离最小的向量替换它，重复 h*w 次，即得替换后的嵌入$z_q$

<div align="center">
    <img src="./imgs/codebook_example.png" alt="codebook_example" style="width: 600px; height: auto;">
</div>

但是这带来了一个新的问题: 梯度怎么从$z_q(x)$传递到$z_e(x)$呢？从数组中取值的操作无法求导，接下来的部分就是解决方案


In [1]:
import torch 
import torch.nn as nn


class ResidualBlock(nn.Module):
    def __init__(self, input_dim:int):
        super().__init__()

        self.relu = nn.ReLU()
        self.conv1 = nn.Conv2d(input_dim, input_dim, 3, 1, 1)
        self.conv2 = nn.Conv2d(input_dim, input_dim, 1)

    def forward(self, x:torch.Tensor):
        # 形状始终为[bs, input_dim, h, w]
        # 由于模型比较简单，可以不使用batch_norm
        x_1 = self.conv1(self.relu(x)) 
        x_2 = self.conv2(self.relu(x_1))

        return x + x_2

class VQVAE(nn.Module):
    def __init__(self, input_dim:int, hidden_dim:int, num_embedding:int):
        super().__init__()

        # input: [bs, input_dim, h, w]
        self.encoder = nn.Sequential(
            nn.Conv2d(input_dim, hidden_dim, 4, 2, 1),  # -> [bs, hidden_dim, h/2, w/2]
            nn.ReLU(),
            nn.Conv2d(hidden_dim, hidden_dim, 4, 2, 1),  # -> [bs, hidden_dim, h/4, w/4]
            nn.ReLU(),
            nn.Conv2d(hidden_dim, hidden_dim, 3, 1, 1),  # 不变
            ResidualBlock(hidden_dim), ResidualBlock(hidden_dim)  # 不变
        )

        # 使用均匀分布初始化codebook
        self.vq_embedding = nn.Embedding(num_embedding, hidden_dim)
        self.vq_embedding.weight.data.uniform_(-1.0/num_embedding, 1.0/num_embedding)

        # input: [bs, hidden_dim, h/4, w/4]
        self.decoder = nn.Sequential(
            nn.Conv2d(hidden_dim, hidden_dim, 3, 1, 1),  # 不变
            ResidualBlock(hidden_dim), ResidualBlock(hidden_dim),  # 不变
            nn.ConvTranspose2d(hidden_dim, hidden_dim, 4, 2, 1),  # [bs, hidden_dim, h/2, w/2]
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim, input_dim, 4, 2, 1)  # [bs, input_dim, h, w]
        )

        self.num_downsample = 2
        

### 梯度传递

为了优化这个编码器-解码器架构，我们就需要一个重建损失进行优化:

$$
\mathcal{L}_{recon} = \parallel x - decoder(z_q(x)) \parallel_2^2 \tag{1}
$$

但是从$z_e(x)$到$z_q(x)$这一步不可导，就无法使用反向传播算法优化了，所以我们使用了一个叫做 **stop gradient** 的运算:

$$
sg(x) = 
\begin{cases}
x & \text{(向前传播)} 
\\
0 & \text{(反向传播)}
\end{cases}
$$

也就是说，向前传播时正常运算，反向传播计算梯度时就要按0来求导，即没有梯度，基于它，就有了下面的式子:

$$
\mathcal{L}_{recon} = \parallel x - decoder(z_e(x) + sg(z_q(x) - z_e(x))) \parallel_2^2 \tag{2}
$$

这样，向前传播时还是式子(1)，但是反向传播时就按照下面的式子求梯度:

$$
\mathcal{L}_{recon} = \parallel x - decoder(z_e(x)) \parallel_2^2 \tag{3}
$$

梯度就可以正常传递到编码器，我们就能使用反向传播进行优化了

综合前两个部分，我们就可以实现一下 forward 函数:

In [2]:
def forward(self, x:torch.Tensor):
        # [bs, input_dim, h, w] -> [bs, hidden_dim, h/4, w/4]
        ze = self.encoder(x)
        
        # 使用广播机制计算距离
        embedding = self.vq_embedding.weight.data 
        bs, c, h, w = ze.shape
        k, _ = embedding.shape
        embedding_broadcast = embedding.reshape(1, k, c, 1, 1)
        ze_broadcast = ze.reshape(bs, 1, c, h, w)
        """
        运算时都变成了: [bs, k, c, h, w]，我们看其中一个bs的[c, h, w]
        可以理解为，我们要把[h, w]中的每个通道向量c替换为codebook中与它最近的向量
        那就要计算通道向量和codebook中k个向量的距离，取最近的向量替换它
        把所有通道向量复制k份得到[k, c, h, w]，再与codebook计算距离，codebook也复制为[k, c, h, w](复制了h*w份)
        计算欧氏距离得到[k, c, h, w]，对c这个维度求和得到[k, h, w]，意思是[h, w]个特征向量和k个codebook向量的距离
        对k通道取min得到每个位置需要的codebook向量的索引，取出之后形状就变回了[h, w, c]
        转置一下就得到替换后的结果[c, h, w]
        """
        distance = torch.sum((embedding_broadcast - ze_broadcast)**2, 2) # -> [bs, k, c, h, w] -> [bs, k, h, w]
        nearest_neighbor = torch.argmin(distance, 1)  # -> [bs, h, w]
        zq = self.vq_embedding(nearest_neighbor).permute(0, 3, 1, 2)  # -> [bs, h, w, c] -> [bs, c, h, w]
        decoder_input = ze + (zq - ze).detach()  # stop gradient

        x_hat = self.decoder(decoder_input)

        return x_hat, ze, zq


VQVAE.forward = forward
vqvae_1 = VQVAE(input_dim=3, hidden_dim=128, num_embedding=32)
x_1 = torch.randn(4, 3, 64, 64)
print(f'Input img size: {x_1.size()}')
x_1_hat, ze, zq = vqvae_1.forward(x_1)
print(f'After VQVAE, x_hat size: {x_1_hat.size()}')
print(f'After VQVAE, ze size: {ze.size()}')
print(f'After VQVAE, zq size: {zq.size()}')

Input img size: torch.Size([4, 3, 64, 64])
After VQVAE, x_hat size: torch.Size([4, 3, 64, 64])
After VQVAE, ze size: torch.Size([4, 128, 16, 16])
After VQVAE, zq size: torch.Size([4, 128, 16, 16])


## 优化嵌入空间

除了优化重建损失，我们还要优化嵌入空间(即 codebook)，因为上面的 stop gradient 把$z_q(x)$的梯度置为了0，codebook 就不能得到反向传播的优化了，所以作者还引入了 码本损失 和 承诺损失:

$$
\mathcal{L}_{codebook} = \parallel sg(z_e(x)) - z_q(x) \parallel_2^2 \quad (这个只优化了 codebook)
\\
\mathcal{L}_{commit} = \parallel z_e(x) - sg(z_q(x)) \parallel_2^2 \quad (这个只优化了编码器)
$$

之所以还要使用 stop gradient 分离编码器和 codebook 的优化，是因为作者认为二者的学习速度不应该一致

那么最后的损失函数如下所示:

$$
\mathcal{L} = \mathcal{L}_{recon} + 
\lambda_1 \cdot \mathcal{L}_{codebook} + 
\lambda_2 \cdot \mathcal{L}_{commit}
$$


In [3]:
def vqvae_loss(x:torch.Tensor, x_hat:torch.Tensor, z_e:torch.Tensor, z_q:torch.Tensor, 
               l_w_embedding = 1, l_w_commitment = 0.25):
    mse_loss = nn.MSELoss()

    l_reconstruct = mse_loss(x, x_hat)  # 重建损失
    l_embedding = mse_loss(ze.detach(), zq)  # 码本学习损失
    l_commitment = mse_loss(ze, zq.detach())  # 承诺损失
    loss = l_reconstruct + \
        l_w_embedding * l_embedding + l_w_commitment * l_commitment
    
    return loss


print(f'VQVAE loss: {vqvae_loss(x_1, x_1_hat, ze, zq)}')


VQVAE loss: 1.024338722229004


## PixelCNN 的适配

我们已经介绍过了 [Gate PixelCNN](./PixelCNN.ipynb)，简而言之，他就是一个自回归生成图像的模型，生成当前像素的值时，只能看到之前已生成的像素；

为了适配 VQVAE，我们需要把 hidden_dim 设置为和 VQVAE 一样的 hidden_dim，并且为了建模离散编码，需要引入一个嵌入层，把输入的形状从[bs, h, w] (也就是每个图像位置需要的 codebook 的向量索引)变成[bs, hidden_dim, h, w]，在参与 Gated PixelCNN 的运算过程；这里的 color_level 就是 codebook 的嵌入数量

In [4]:
from c_vqvae import GatedBlock, GatedPixelCNN


class PixelCNNWithEmbedding(GatedPixelCNN):
    def __init__(self, n_blocks, hidden_dim, linear_dim, bn=True, color_level=256):
        # 这里我封装一下，原始代码里用的p，为了清晰，这里我用hidden_dim替代一下
        # color_level在这里就是num_embedding
        super().__init__(n_blocks, hidden_dim, linear_dim, bn, color_level)

        self.embedding = nn.Embedding(color_level, hidden_dim)
        self.block1 = GatedBlock('A', hidden_dim, hidden_dim, bn)

    def forward(self, x:torch.Tensor):
        # input: [bs, h, w]
        x = self.embedding(x)  # -> [bs, h, w, hidden_dim]
        x = x.permute(0, 3, 1, 2).contiguous()  # -> [bs, hidden_dim, h, w]

        return super().forward(x)  # -> [bs, color_level, h, w]


In [7]:
pixelcnn = PixelCNNWithEmbedding(
    n_blocks=15,
    hidden_dim=128,
    linear_dim=256,
    bn=True,
    color_level=32
)
x_2 = torch.zeros(4, 64, 64).to(torch.long)

print(f'Input size: {x_2.size()}')
output_2 = pixelcnn(x_2)
print(f'Output size: {output_2.size()}')

Input size: torch.Size([4, 64, 64])
Output size: torch.Size([4, 32, 64, 64])


## 采样

采样过程其实也很明了了: 先生成[bs, h, w]的索引图，每个索引表示需要的 codebook 中的向量的索引，然后使用 PixelCNN 自回归采样生成最后的[bs, h, w]的索引图，再送入解码器函数转换得到$z_q(x)$后进行解码得到最终的图像[bs, c, h, w]

In [11]:
from torchvision.utils import save_image
import torch.nn.functional as F
import os


def sample(vqvae_model:VQVAE, pixelcnn_model:PixelCNNWithEmbedding, sample_num = 4, 
           sample_save_dir = "./sample", img_size = 64):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    vqvae_model = vqvae_model.to(device)
    pixelcnn_model = pixelcnn_model.to(device)
    vqvae_model.eval()
    pixelcnn_model.eval()

    os.makedirs(sample_save_dir, exist_ok=True)

    h, w = vqvae_model.get_latent_hw((sample_num, img_size, img_size))
    input_shape = (sample_num, h, w)
    x = torch.zeros(input_shape).to(device).to(torch.long)

    with torch.no_grad():
            for i in range(h):
                for j in range(w):
                    output = pixelcnn_model(x)  # -> [bs, num_embedding, h, w]
                    # -> [bs, num_embedding]，位置(i, j)的概率分布(softmax logits转换而来)
                    next_pixel_prob = F.softmax(output[:, :, i, j], dim=-1)
                    next_pixel = torch.multinomial(next_pixel_prob, num_samples=1)  # 采样 -> [bs, 1]
                    x[:, i, j] = next_pixel[:, 0]

    # range(-1, 1) -> range(0, 1)
    x_hat = vqvae_model.decode(x)
    x_hat = torch.clamp(x_hat, -1.0, 1.0)
    x_hat = (x_hat + 1.0) / 2.0

    for i in range(x_hat.shape[0]):
        img_recon = x_hat[i].cpu()
        save_recon_path = os.path.join(sample_save_dir, f"sample_img{i}.png")
        save_image(img_recon, save_recon_path)

        print(f'Save sample img at: {save_recon_path}')


## 训练示例

### 训练 VQVAE

In [8]:
from c_vqvae import (
    get_config, get_vqvae, get_pixelcnn_test, get_pixelcnn,
    train_vqvae, show_train_vqvae_recon, train_pixelcnn,
    sample
)


config = get_config('./configs/vqvae_config.yaml')
vqvae = get_vqvae(config)
train_vqvae_config = config["train_vqvae"]

train_vqvae(
    vqvae_model=vqvae,
    img_size=config["global"]["img_size"],
    checkpoint_save_dir=train_vqvae_config["checkpoint_save_dir"],
    batch_size=train_vqvae_config["batch_size"],
    lr=train_vqvae_config["lr"],
    epochs=train_vqvae_config["epochs"],
    l_w_embedding=train_vqvae_config["l_w_embedding"],
    l_w_commitment=train_vqvae_config["l_w_commitment"],
    dataset_dir=train_vqvae_config["dataset_dir"]
)


Training: 100%|██████████| 500/500 [00:13<00:00, 37.65it/s, avg_loss=0.00613]

[trainer]Saved: vqvae_iter500.pth


In [9]:
# 使用预训练的VQVAE进行一次测试，重建输入图像
config = get_config('./configs/vqvae_config.yaml')
train_vqvae_config = config["train_vqvae"]
vqvae = get_vqvae(config, load_pretrained=True)

show_train_vqvae_recon(
    vqvae_model=vqvae,
    dataset_dir=train_vqvae_config["dataset_dir"],
    save_res_dir=train_vqvae_config["save_res_dir"]
)


[VQVAE]Loaded checkpoint from ./ckpts/vqvae/vqvae_iter5000.pth


Testing VQVAE recon: 100%|██████████| 1/1 [00:00<00:00, 24.22it/s]

Save recon img at: ./vqvae_recon\batch0_img0_recon.png
Save recon img at: ./vqvae_recon\batch0_img1_recon.png


重建结果如下:

<div align="center" style="display: flex; justify-content: center; gap: 20px;">
    <div>
        <img src="./vqvae_recon/batch0_img0_recon.png" alt="img0" style="width: 100px; height: auto;">
    </div>
    <div>
        <img src="./vqvae_recon/batch0_img1_recon.png" alt="img1" style="width: 100px; height: auto;">
    </div>
</div>

### 训练 PixelCNN

In [10]:
config = get_config('./configs/vqvae_config.yaml')
train_pixelcnn_config = config["train_pixelcnn"]
vqvae = get_vqvae(config, load_pretrained=True)
pixelcnn = get_pixelcnn(config)

train_pixelcnn(
    vqvae_model=vqvae,
    pixelcnn_model=pixelcnn,
    img_size=config["global"]["img_size"],
    checkpoint_save_dir=train_pixelcnn_config["checkpoint_save_dir"],
    batch_size=train_pixelcnn_config["batch_size"],
    lr=train_pixelcnn_config["lr"],
    epochs=train_pixelcnn_config["epochs"],
    dataset_dir=train_pixelcnn_config["dataset_dir"]
)


[VQVAE]Loaded checkpoint from ./ckpts/vqvae/vqvae_iter5000.pth


Training: 100%|██████████| 500/500 [00:32<00:00, 15.46it/s, avg_loss=0.00758]


[trainer]Saved: pixelcnn_iter500.pth


## 测试示例

In [12]:
config = get_config('./configs/vqvae_config.yaml')
sample_config = config["sample"]
vqvae = get_vqvae(config, load_pretrained=True)
pixelcnn = get_pixelcnn(config, load_pretrained=True)

sample(
    vqvae_model=vqvae,
    pixelcnn_model=pixelcnn,
    sample_num=sample_config["sample_num"],
    sample_save_dir=sample_config["sample_save_dir"],
    img_size=config["global"]["img_size"]
)


[VQVAE]Loaded checkpoint from ./ckpts/vqvae/vqvae_iter5000.pth
[PixelCNN]Loaded checkpoint from ./ckpts/pixelcnn/pixelcnn_iter5000.pth
Save sample img at: ./sample\sample_img0.png
Save sample img at: ./sample\sample_img1.png
Save sample img at: ./sample\sample_img2.png
Save sample img at: ./sample\sample_img3.png


采样结果如下:

<div align="center" style="display: flex; justify-content: center; gap: 20px;">
    <div>
        <img src="./sample\sample_img0.png" alt="img0" style="width: 100px; height: auto;">
    </div>
    <div>
        <img src="./sample\sample_img1.png" alt="img1" style="width: 100px; height: auto;">
    </div>
    <div>
        <img src="./sample\sample_img2.png" alt="img2" style="width: 100px; height: auto;">
    </div>
    <div>
        <img src="./sample\sample_img3.png" alt="img3" style="width: 100px; height: auto;">
    </div>
</div>